# Intro
Intro basée sur l'article, dire que les ramsomware c'est un type de virus, et que les transactions bitcoin ont permis de rendre le truc anonyme et plus facile. Donc faudrait être en mesure de les détecter avant.
Etat de l'art c'est que il n'y a que des méthodes heuristiques pour le moment, et faudrait faire des méthodes de machine learning.


In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd
from datetime import datetime, timezone

from IPython.display import display

In [2]:
data = pd.read_csv("BitcoinHeistData.csv")

In [3]:
data.head()

,address,year,day,length,weight,count,looped,neighbors,income,label
0,111K8kZAEnJg245r2cM6y9zgJGHZtJPy6,2017,11,18,0.008333,1,0,2,100050000.0,princetonCerber
1,1123pJv8jzeFQaCV4w644pzQJzVWay2zcA,2016,132,44,0.000244,1,0,1,100000000.0,princetonLocky
2,112536im7hy6wtKbpH1qYDWtTyMRAcA2p7,2016,246,0,1.000000,1,0,2,200000000.0,princetonCerber
3,1126eDRw2wqSkWosjTCre8cjjQW8sSeWH7,2016,322,72,0.003906,1,0,2,71200000.0,princetonCerber
4,1129TSjKtx65E35GiUo4AYVeyo48twbrGX,2016,238,144,0.072848,456,0,1,200000000.0,princetonLocky


In [ ]:
features = data.columns[:-1].to_list()  # Exclude the target column
features

['address',
 'year',
 'day',
 'length',
 'weight',
 'count',
 'looped',
 'neighbors',
 'income']

In [ ]:
behavior_features = features[3:]
numerical_features = features[1:]
integer_cols = set(numerical_features) - {"weight"}


## Description des variables

Chaque ligne décrit une adresse Bitcoin et les caractéristiques de son activité dans le graphe de transactions.

| Variable | Description |
|---|---|
| `address` | Identifiant public de l'adresse Bitcoin. Ce n'est pas une mesure numérique : il est généralement préférable de ne pas l'utiliser directement comme entrée d'un modèle. |
| `year` | Année de l'observation. Elle situe l'activité dans le temps et peut refléter des changements historiques dans les campagnes ou les pratiques de paiement. |
| `day` | Jour de l'année, de 1 à 365 (ou 366). Par exemple, `132` est le 132e jour de l'année, et non le 132e jour du mois. |
| `length` | Longueur du chemin ou de la chaîne de transactions associée à l'adresse, mesurée en nombre d'étapes. Une valeur élevée indique un parcours plus long et plus complexe. |
| `weight` | Poids relatif de la chaîne ou du parcours étudié dans le graphe. C'est une proportion sans unité, généralement comprise entre 0 et 1 : une valeur proche de 1 indique une chaîne dominante. |
| `count` | Nombre total de transactions associées à l'adresse dans le graphe étudié. Il mesure l'activité transactionnelle, mais ne suffit pas à lui seul à identifier une adresse malveillante. |
| `looped` | Nombre de transactions qui participent à des boucles dans le graphe. Les boucles correspondent à des flux qui reviennent vers une adresse ou un ensemble d'adresses déjà rencontré. |
| `neighbors` | Nombre d'adresses directement connectées à l'adresse par une transaction entrante ou sortante. Il s'agit du degré local du nœud dans le graphe Bitcoin. |
| `income` | Montant total reçu par l'adresse, exprimé en satoshis. `100 000 000` satoshis = `1 BTC`; ainsi `100050000.0` correspond à `1.0005 BTC`. |
| `label` | Classe cible à prédire, par exemple `princetonCerber` ou `princetonLocky`. Elle n'est pas une feature : elle sert de variable de sortie (`target`). |

### Pour la modélisation :

- `address` est un identifiant et ne possède pas de valeur concrète ; il faut l'exclure ou l'encoder.
- `label` ne doit pas être utilisée comme entrée.
- `length`, `count`, `looped` et `neighbors` sont des comptages, `weight` est une proportion et `income` est un montant en satoshis. Une standardisation ou une transformation logarithmique peut être utile pour certains modèles.
- `year` et `day` sont temporelles ; `day` peut être transformée en variable cycliques afin de représenter la périodicité du calendrier.


In [13]:
labels = data["label"].values  # Assuming the target column is named "Target"
types_of_labels = np.unique(labels)
types_of_labels

array(['montrealAPT', 'montrealComradeCircle', 'montrealCryptConsole',
       'montrealCryptXXX', 'montrealCryptoLocker',
       'montrealCryptoTorLocker2015', 'montrealDMALocker',
       'montrealDMALockerv3', 'montrealEDA2', 'montrealFlyper',
       'montrealGlobe', 'montrealGlobeImposter', 'montrealGlobev3',
       'montrealJigSaw', 'montrealNoobCrypt', 'montrealRazy',
       'montrealSam', 'montrealSamSam', 'montrealVenusLocker',
       'montrealWannaCry', 'montrealXLocker', 'montrealXLockerv5.0',
       'montrealXTPLocker', 'paduaCryptoWall', 'paduaJigsaw',
       'paduaKeRanger', 'princetonCerber', 'princetonLocky', 'white'],
      dtype=object)

`white` correspond à une adresse non connue comme un ransomware. Attention, ce n'est pas un innoncent confirmé. Le but de cette analyse sera de détecter quelles adresses avec un label `white` sont malveillantes, basé sur leur activité précédente.

In [16]:
required_cols = features + ["label"]
missing_cols = set(required_cols) - set(data.columns)
if missing_cols:
    raise ValueError(f"Colonnes absentes : {sorted(missing_cols)}")

if not data.columns.is_unique:
    raise ValueError("Certains noms de colonnes sont dupliqués.")


-Il n'y a aucune valeur manquante dans le dataset, ni de doublon

In [20]:
# data reste intact. Les index de work correspondent aux positions dans data. On créée aussi un dataframe avec les valeurs invalides à la fin. False siginifique que tout est considéré valide au début
work = data.reset_index(drop=True).copy() 
invalid = pd.DataFrame(
    False, index=work.index, columns=numerical_features
)

parsed = {}
current_year = datetime.now(timezone.utc).year
for ft in numerical_features:
    col = work[ft]
    values = pd.to_numeric(col, errors="coerce") #on convertit en nombre (au cas où c'est stocké en string). "coerce" permet de transformer ce qui ne va pas en "NaN"

    finite = pd.Series(np.isfinite(values), index=work.index).fillna(False) #assure qu'on ne garde que des valeurs finies, avec le même index que notre dataset

    bad = (col.notna()) & (~finite) | values.lt(0).fillna(False)  #enlève les invalides (NaN, négatives, finies)
    if ft in integer_cols:
        bad  |= (finite & values.mod(1).ne(0)).fillna(False) #entier différent de 0. Les Nan sont remplacés par False
    if ft == "year":
        bad |= finite & ~values.between(2009, current_year)
    if ft == "day":
        bad |= finite & ~values.between(1, 366)

    invalid[ft] = bad.fillna(False)
    parsed[ft] = values.where(finite)

numeric = pd.DataFrame(parsed)
fixed_year = numeric["year"].mask(invalid["year"]).astype("Int64")
fixed_day = numeric["day"].mask(invalid["day"]).astype("Int64")

start = pd.to_datetime(fixed_year.astype(str),format = "%Y", errors="coerce") #convertit une année en la date du premier janvier (ex 2017 --> 2017-01-01)
date = start + pd.to_timedelta(fixed_day - 1, unit="D")

bad_calendar = (date.notna() & date.dt.year.ne(fixed_year).fillna(False)) #en ajoutant un an à une année on passe à la suivante (vérifie les années bisextiles entre autres)
invalid["day"] |= bad_calendar
date = date.mask(invalid["day"]|invalid["year"]) 


rows = []
for col in work.columns:
    s = work[col]
    row = {"variable": col,
            "type_original": str(s.dtype),
            "nb_lignes": len(s),
            "nb_manquantes": int(s.isna().sum()),
            "%_manquant": 100 * s.isna().mean(),
            "nb_distinctes": s.nunique(dropna=True),
            "nb_chaines_vides": (
                int(s.astype("string").str.strip().eq("").sum())
                if col not in numerical_features else 0
            ),
            "nb_invalides": (
                int(invalid[col].sum()) if col in numerical_features else 0
            ),
        }
    if col in numerical_features:
        row["nb_zeros"] = int(numeric[col].eq(0).sum())
        row["moyenne"] = numeric[col].mean()
        row["ecart_type"] = numeric[col].std()
    rows.append(row)

quality_table = pd.DataFrame(rows).set_index("variable")
quantiles = numeric.quantile([0,0.01,0.25,0.5,0.75,0.99,1]).T
quantiles.columns = ["min", "p01", "p25", "mediane", "p75", "p99", "max"]
quality_table = quality_table.join(quantiles)

display(quality_table.round(3))
print(f"Lignes : {len(work):,}")
print(f"Adresses distinctes : {work['address'].nunique():,}")
print(f"Doublons stricts supplémentaires : {work.duplicated().sum():,}")

display(
    work["label"]
    .value_counts(dropna=False)
    .rename("nb_lignes")
    .to_frame()
)

#exemples d'anomalies
display(work.loc[invalid.any(axis=1), required_cols].head(20))


,type_original,nb_lignes,nb_manquantes,%_manquant,nb_distinctes,nb_chaines_vides,nb_invalides,nb_zeros,moyenne,ecart_type,min,p01,p25,mediane,p75,p99,max
variable,,,,,,,,,,,,,,,,,
address,str,2916697,0,0.0,2631095,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year,int64,2916697,0,0.0,8,0,0,0.0,2.014475e+03,2.257000e+00,2011.0,2011.0,2.013000e+03,2.014000e+03,2.016000e+03,2.018000e+03,2.018000e+03
day,int64,2916697,0,0.0,365,0,0,0.0,1.814570e+02,1.040120e+02,1.0,4.0,9.200000e+01,1.810000e+02,2.710000e+02,3.610000e+02,3.650000e+02
length,int64,2916697,0,0.0,73,0,0,708621.0,4.500900e+01,5.898200e+01,0.0,0.0,2.000000e+00,8.000000e+00,1.080000e+02,1.440000e+02,1.440000e+02
weight,float64,2916697,0,0.0,784927,0,0,0.0,5.460000e-01,3.674000e+00,0.0,0.0,2.100000e-02,2.500000e-01,8.820000e-01,3.417000e+00,1.943749e+03
count,int64,2916697,0,0.0,11572,0,0,0.0,7.216450e+02,1.689676e+03,1.0,1.0,1.000000e+00,1.000000e+00,5.600000e+01,7.725000e+03,1.449700e+04
looped,int64,2916697,0,0.0,10168,0,0,2504877.0,2.385070e+02,9.663220e+02,0.0,0.0,0.000000e+00,0.000000e+00,0.000000e+00,5.391000e+03,1.449600e+04
neighbors,int64,2916697,0,0.0,814,0,0,0.0,2.207000e+00,1.791900e+01,1.0,1.0,1.000000e+00,2.000000e+00,2.000000e+00,8.000000e+00,1.292000e+04
income,float64,2916697,0,0.0,1866365,0,0,0.0,4.464889e+09,1.626860e+11,30000000.0,30500000.0,7.428559e+07,1.999985e+08,9.940000e+08,4.642012e+10,4.996440e+13


Lignes : 2,916,697
Adresses distinctes : 2,631,095
Doublons stricts supplémentaires : 0


,nb_lignes
label,
white,2875284
paduaCryptoWall,12390
montrealCryptoLocker,9315
princetonCerber,9223
princetonLocky,6625
montrealCryptXXX,2419
montrealNoobCrypt,483
montrealDMALockerv3,354
montrealDMALocker,251


,address,year,day,length,weight,count,looped,neighbors,income,label
